### Handling PDF Challenges 
PDFs are notoriously difficult to parse because they:

- Store text in complex ways (not just simple text)
- Can have formatting issues
- May contain scanned images (requiring OCR)
- Often have extraction artifacts


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from typing import List

In [12]:
# Example of raw PDF extraction
raw_pdf_text = """Company Financial Report


    The ﬁnancial performance for ﬁscal year 2025
    shows signiﬁcant growth in proﬁtability.
    
    
    
    Revenue increased by 25%.
    
The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""

# Apply the cleaning function
def clean_text(text):
    # Remove excessive whitespace
    text = " ".join(text.split())
    
    # Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")
    
    return text

cleaned = clean_text(raw_pdf_text)
print("BEFORE:")
print(repr(raw_pdf_text))
print("\nAFTER:")
print(repr(cleaned))

# Output:
# BEFORE:
# 'Company Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2025\n    shows signiﬁcan'
# 
# AFTER:
# 'Company Financial Report The financial performance for fiscal year 2025 shows significant growth in'

BEFORE:
"Company Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2025\n    shows signiﬁcant growth in proﬁtability.\n\n\n\n    Revenue increased by 25%.\n\nThe company's efﬁciency improved due to workﬂow\noptimization.\n\n\nPage 1 of 10\n"

AFTER:
"Company Financial Report The financial performance for fiscal year 2025 shows significant growth in profitability. Revenue increased by 25%. The company's efficiency improved due to workflow optimization. Page 1 of 10"


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_core.documents import Document
class SmartPDFProcessor:
    "Advanced PDF processing with error handling and text cleaning"
    def __init__(self, chunk_size=1000, chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, 
            chunk_overlap=chunk_overlap,
            separators=[" "],
        )
    
    def process_pdf(self, file_path: str) -> List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""
        
        # Load PDF
        loader = PyPDFLoader(file_path)
        pages = loader.load()

        ## Process each page
        processed_chunks = []

        for page_num, page in enumerate(pages):
            # Clean text
            cleaned_text = self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text) < 30:
                continue

            # Split into smart chunks with metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)

                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks
    
    def _clean_text(self, text: str) -> str:
        """Clean text by removing excessive whitespace and fixing common OCR issues"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common ligatures
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

In [15]:
preprocessor = SmartPDFProcessor()

In [20]:
### Process a PDF if available
try:
    smart_chunks = preprocessor.process_pdf("./attention.pdf")
    print(f"Processed {len(smart_chunks)} smart chunks from PDF.")

    if smart_chunks:
        for key, value in smart_chunks[0].metadata.items():
            print(f"{key}: {value}")
except FileNotFoundError:
    print("PDF file not found. Please check the file path.")

Processed 52 smart chunks from PDF.
producer: pdfTeX-1.40.25
creator: LaTeX with hyperref
creationdate: 2023-08-03T00:07:29+00:00
author: 
keywords: 
moddate: 2023-08-03T00:07:29+00:00
ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
subject: 
title: 
trapped: /False
source: ./attention.pdf
total_pages: 15
page: 1
page_label: 1
chunk_method: smart_pdf_processor
char_count: 2858
